# Session 5: Evaluating Models — The Full Picture

## The Story

Your fraud team built a model. They claim it's "90% accurate." The CFO wants to know: **how much money will it save?**

"Accuracy" doesn't answer that question. This notebook walks through the tools that do: confusion matrices, ROC curves, and profit analysis.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, 
                             precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# Simulate fraud detection dataset
n = 20000
fraud_rate = 0.04  # 4% fraud

data = pd.DataFrame({
    'claim_amount': np.random.lognormal(8.5, 1.0, n).clip(500, 100000).astype(int),
    'policy_age_months': np.random.exponential(36, n).clip(1, 240).astype(int),
    'claimant_age': np.random.normal(42, 14, n).clip(18, 80).astype(int),
    'previous_claims': np.random.poisson(0.8, n),
    'days_to_report': np.random.exponential(5, n).clip(0, 60).astype(int),
    'involves_bodily_injury': np.random.choice([0, 1], n, p=[0.7, 0.3]),
    'weekend_incident': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    'third_party_involved': np.random.choice([0, 1], n, p=[0.5, 0.5]),
})

# Fraud depends on realistic risk factors
fraud_score = (
    0.25 * (data['claim_amount'] > 15000).astype(float) +
    0.2 * (data['policy_age_months'] < 6).astype(float) +
    0.15 * (data['previous_claims'] >= 3).astype(float) +
    0.15 * (data['days_to_report'] > 14).astype(float) +
    0.1 * data['involves_bodily_injury'] +
    0.1 * (data['claimant_age'] < 28).astype(float) +
    0.05 * data['weekend_incident'] +
    np.random.normal(0, 0.25, n)
)
data['is_fraud'] = (fraud_score > np.percentile(fraud_score, 100 - fraud_rate * 100)).astype(int)

features = data.columns[:-1].tolist()
X = data[features]
y = data['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Build model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

print("📋 FRAUD DETECTION SETUP")
print("=" * 50)
print(f"  Claims: {n:,} (training: {len(X_train):,}, test: {len(X_test):,})")
print(f"  Fraud rate: {y.mean():.1%} ({y.sum():,} cases)")
print(f"  Average claim: €{data['claim_amount'].mean():,.0f}")
print(f"  Average fraud claim: €{data[data['is_fraud']==1]['claim_amount'].mean():,.0f}")

## Chapter 1: The Accuracy Trap

The team says: **"Our model is 96% accurate!"**

In [ ]:
# The accuracy trap
from sklearn.metrics import accuracy_score

# Model accuracy
y_pred = model.predict(X_test)
model_accuracy = accuracy_score(y_test, y_pred)

# "Always predict not fraud" baseline
baseline_accuracy = 1 - y_test.mean()

print("📊 THE ACCURACY TRAP")
print("=" * 50)
print(f"\n  Our model accuracy:                    {model_accuracy:.1%}")
print(f"  'Always predict not fraud' accuracy:    {baseline_accuracy:.1%}")
print(f"  \n  Difference:                            {model_accuracy - baseline_accuracy:.1%}")
print(f"\n  ⚠️ A model that predicts 'not fraud' for EVERY claim")
print(f"     is already {baseline_accuracy:.1%} 'accurate' — and catches ZERO fraud.")
print(f"\n  💡 Accuracy is meaningless with imbalanced data.")
print(f"     We need to look at the TYPES of errors, not just the total.")

## Chapter 2: The Confusion Matrix

Four types of outcomes, each with a different business impact:

In [ ]:
# Confusion matrix at default threshold (0.5)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("📊 CONFUSION MATRIX (threshold = 0.5)")
print("=" * 55)
print(f"\n                        Actually Fraud    Actually Legit")
print(f"  Predicted Fraud       {tp:>8,} (TP)      {fp:>8,} (FP)")
print(f"  Predicted Legit       {fn:>8,} (FN)      {tn:>8,} (TN)")

print(f"\n  INTERPRETATION:")
print(f"    ✅ True Positives:   {tp:,} fraud cases correctly caught")
print(f"    ❌ False Positives:  {fp:,} legitimate claims wrongly flagged")
print(f"    ❌ False Negatives:  {fn:,} fraud cases MISSED")
print(f"    ✅ True Negatives:   {tn:,} legitimate claims correctly approved")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn)

print(f"\n  Precision: {precision:.1%} — 'Of those we flagged, {precision:.0%} were actual fraud'")
print(f"  Recall:    {recall:.1%} — 'We caught {recall:.0%} of all fraud cases'")

## Chapter 3: The Threshold — Your Business Decision

In [ ]:
# What happens at different thresholds?
thresholds = [0.05, 0.10, 0.20, 0.30, 0.50, 0.70]

print("📊 PERFORMANCE AT DIFFERENT THRESHOLDS")
print("=" * 80)
print(f"\n  {'Threshold':<12} {'Flagged':<10} {'TP':<8} {'FP':<8} {'FN':<8} {'Precision':<12} {'Recall':<12}")
print("  " + "-" * 75)

for t in thresholds:
    pred = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    prec = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    flagged = pred.sum()
    print(f"  {t:<12.2f} {flagged:<10} {tp_t:<8} {fp_t:<8} {fn_t:<8} {prec:<12.1%} {rec:<12.1%}")

print(f"\n  💡 Low threshold (0.05): catch most fraud, but flag MANY legitimate claims")
print(f"     High threshold (0.70): very few false alarms, but miss most fraud")
print(f"\n     Which is right? It depends on YOUR cost structure.")

## Chapter 4: The ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#00008f', lw=2.5, label=f'Our Model (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', lw=1.5, label='Random Guess (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#00008f')

# Mark a specific operating point
operating_idx = np.argmin(np.abs(roc_thresholds - 0.15))
ax.scatter(fpr[operating_idx], tpr[operating_idx], color='#FF1721', s=150, zorder=5,
           label=f'Operating point (t=0.15)', edgecolors='black', linewidths=1.5)

ax.set_xlabel('False Positive Rate (false alarms)', fontsize=12)
ax.set_ylabel('True Positive Rate (fraud caught)', fontsize=12)
ax.set_title('ROC Curve: The Trade-off Between Catch Rate and False Alarms', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"  AUC = {auc:.3f}")
print(f"  → The model is {'significantly' if auc > 0.75 else 'moderately'} better than random.")
print(f"  → The red dot shows our chosen operating point.")
print(f"     Moving it left = fewer false alarms but less fraud caught.")
print(f"     Moving it right = more fraud caught but more false alarms.")

## Chapter 5: The Expected Value Framework — Money Talks

In [ ]:
# Expected value analysis
avg_fraud_amount = data[data['is_fraud'] == 1]['claim_amount'].mean()
investigation_cost = 600  # Cost per flagged claim

print("📊 EXPECTED VALUE ANALYSIS")
print("=" * 70)
print(f"\n  Cost assumptions:")
print(f"    Average fraudulent claim: €{avg_fraud_amount:,.0f}")
print(f"    Investigation cost per flag: €{investigation_cost:,}")
print(f"    Cost of missing fraud: €{avg_fraud_amount:,.0f} (full payout)")

# Scale to annual portfolio
annual_claims = 50000
scale = annual_claims / len(y_test)

print(f"\n  Projecting to annual portfolio of {annual_claims:,} claims:")
print(f"\n  {'Threshold':<12} {'Fraud Saved':<18} {'Investigation':<18} {'Fraud Missed':<18} {'NET VALUE'}")
print("  " + "-" * 80)

best_net = -np.inf
best_threshold = 0

for t in [0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    pred = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    
    saved = tp_t * scale * avg_fraud_amount
    invest = (tp_t + fp_t) * scale * investigation_cost
    missed = fn_t * scale * avg_fraud_amount
    net = saved - invest
    
    if net > best_net:
        best_net = net
        best_threshold = t
    
    print(f"  {t:<12.2f} €{saved:>14,.0f}   €{invest:>14,.0f}   €{missed:>14,.0f}   €{net:>12,.0f}")

print(f"\n  ✅ OPTIMAL THRESHOLD: {best_threshold} → Net annual value: €{best_net:,.0f}")
print(f"\n  💡 Notice: the best FINANCIAL threshold is NOT the best ACCURACY threshold.")
print(f"     We optimize for EUROS, not percentages.")

In [ ]:
# Profit curve
profits = []
threshold_range = np.linspace(0.01, 0.99, 100)

for t in threshold_range:
    pred = (y_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, pred)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    
    profit = (tp_t * avg_fraud_amount - (tp_t + fp_t) * investigation_cost) * scale
    profits.append(profit)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_range, [p / 1e6 for p in profits], color='#00008f', lw=2.5)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Break-even')

# Mark the optimal point
optimal_idx = np.argmax(profits)
ax.scatter(threshold_range[optimal_idx], profits[optimal_idx] / 1e6, 
           color='#FF1721', s=150, zorder=5, edgecolors='black', linewidths=1.5,
           label=f'Optimal: t={threshold_range[optimal_idx]:.2f}, €{profits[optimal_idx]/1e6:.1f}M')

ax.set_xlabel('Threshold (flag if P(fraud) > threshold)', fontsize=12)
ax.set_ylabel('Annual Profit (€ Millions)', fontsize=12)
ax.set_title('Profit Curve: Finding the Optimal Operating Point', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n→ The profit curve peaks at threshold {threshold_range[optimal_idx]:.2f}")
print(f"  Below that: we're investigating too many low-probability cases (cost exceeds savings)")
print(f"  Above that: we're missing too many real fraud cases")
print(f"\n  💡 THIS is the chart your CFO wants to see, not the ROC curve.")

## Chapter 6: Lift — Prioritizing Limited Resources

In [ ]:
# Lift chart: how much of the fraud do we find by investigating top X%?
sorted_indices = np.argsort(-y_prob)  # Sort by model score, highest first
y_sorted = y_test.values[sorted_indices]

cumulative_fraud = np.cumsum(y_sorted) / y_sorted.sum()
pct_investigated = np.arange(1, len(y_sorted) + 1) / len(y_sorted)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(pct_investigated * 100, cumulative_fraud * 100, color='#00008f', lw=2.5, label='Our model')
ax.plot([0, 100], [0, 100], '--', color='gray', lw=1.5, label='Random investigation')

# Highlight key points
for pct in [10, 20, 50]:
    idx = int(len(y_sorted) * pct / 100) - 1
    caught = cumulative_fraud[idx] * 100
    ax.scatter(pct, caught, color='#FF1721', s=100, zorder=5)
    ax.annotate(f'{caught:.0f}% fraud\ncaught', xy=(pct, caught), 
                xytext=(pct + 5, caught - 8), fontsize=10,
                arrowprops=dict(arrowstyle='->', color='#FF1721'))

ax.set_xlabel('% of Claims Investigated (ranked by model score)', fontsize=12)
ax.set_ylabel('% of All Fraud Caught', fontsize=12)
ax.set_title('Lift Chart: Investigate Smarter, Not Harder', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

for pct in [10, 20, 50]:
    idx = int(len(y_sorted) * pct / 100) - 1
    caught = cumulative_fraud[idx] * 100
    lift = caught / pct
    print(f"  Investigate top {pct}% → catch {caught:.0f}% of fraud (lift = {lift:.1f}×)")

print(f"\n  💡 With limited investigators, the model tells you WHERE TO LOOK FIRST.")
print(f"     Instead of random checks, focus on the top 20% by model score.")

---

## Key Takeaways

1. **Accuracy is misleading** — a "97% accurate" model may catch zero fraud
2. **The confusion matrix reveals WHAT errors the model makes** — and they have different costs
3. **The threshold is YOUR decision** — the model gives probabilities, you choose where to act
4. **Expected value = the right metric** — evaluate models in euros, not percentages
5. **The profit curve finds the optimal operating point** — where savings minus costs peaks
6. **Lift shows how to prioritize** — investigate smarter with limited resources

> When your team presents a model, ask: *"Show me the profit curve and the lift chart."*